Language models output text. But there are times where you want to get more structured information than just text back

Output parsers are classes that help structure language model responses. There are two main methods an output parser must implement:

- **Get format instructions**: A method which returns a string containing instructions for how the output of a language model should be formatted.
- **Parse**: A method which takes in a string (assumed to be the response from a language model) and parses it into some structure.

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

In [3]:
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate,
                                        PromptTemplate
                                        )

# Pydantic in LangChain: Notes & Example

---

## 1. What is Pydantic?

- **Pydantic** is a Python library for **data validation** and **settings management** using Python type annotations.
- It provides a `BaseModel` class for declaring data models with types, constraints, and descriptions.
- It **automatically validates** and parses data, raising errors for missing or invalid fields.

**In LangChain, Pydantic models are often used for:**
- Defining structured outputs expected from an LLM (e.g., a Joke object, a product recommendation, etc.).
- Validating and parsing LLM outputs into safe, strongly-typed Python objects.

---

## 2. Declaring a Pydantic Model

```python
from typing import Optional
from pydantic import BaseModel, Field

class Joke(BaseModel):
    """Joke to tell user"""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(description="The rating of the joke is from 1 to 10", default=None)


In [5]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [7]:
?Field

Signature:
Field(
    default: 'Any' = PydanticUndefined,
    *,
    default_factory: 'Callable[[], Any] | Callable[[dict[str, Any]], Any] | None' = PydanticUndefined,
    alias: 'str | None' = PydanticUndefined,
    alias_priority: 'int | None' = PydanticUndefined,
    validation_alias: 'str | AliasPath | AliasChoices | None' = PydanticUndefined,
    serialization_alias: 'str | None' = PydanticUndefined,
    title: 'str | None' = PydanticUndefined,
    field_title_generator: 'Callable[[str, FieldInfo], str] | None' = PydanticUndefined,
    description: 'str | None' = PydanticUndefined,
    examples: 'list[Any] | None' = PydanticUndefined,
    exclude: 'bool | None' = PydanticUndefined,
    discriminator: 'str | types.Discriminator | None' = PydanticUndefined,
    deprecated: 'Deprecated | str | bool | None' = PydanticUndefined,
    json_schema_extra: 'JsonDict | Callable[[JsonDict], None] | None' = PydanticUndefined,
    frozen: 'bool | None' = PydanticUndefined,
    validate_default:

In [8]:
# # #Analogy ---- Create a table named Joke which has 3 columns
# #               setup : str, not null
#                 punchline: str , not null
#                 rating: int, null, None
#
# Constraints for Field Object
#==============================
# ge --- greather than equal to
# le --- less than equal to
# gt --- greather than
# lt --- less than

class Joke(BaseModel):
    """Joke to tell user"""

    setup: str =  Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] =  Field(description="The rating of the joke")

In [10]:
parser =PydanticOutputParser(pydantic_object=Joke)

In [11]:
instruction = parser.get_format_instructions()

In [12]:
instruction = parser.get_format_instructions()

In [13]:
instruction

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Joke to tell user", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "description": "The rating of the joke", "title": "Rating"}}, "required": ["setup", "punchline", "rating"]}\n```'

# Method 1 - Using pydantic object inside prompt (format instructions)

In [17]:
prompt = PromptTemplate(
    # The 'template' argument defines the format of the prompt to be sent to the LLM
    template='''
    Answer the user query with a joke. Here is your formatting instruction.
    {format_instruction}
    
    Query: {query}
    Answer: ''',

    # 'input_variables' specifies which keys the template expects at runtime (here : 'query')
    input_variables =['query'],

    #partial_variables are values you want to "hard-code" or fill in at creation time.
    #Here, format_instruction is filled automatically with  instructions from the parser
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

#The result is a prompt template that, when given a 'query', will fill in both
#'query' (user's actual question) and format_instruction (instructions for output formatting)


#Step2: Create a chain by conncecting the prompt template to the LL<
# this means: the input dict will first fill the prompt, then be sent to the LLM

chain = prompt | llm

In [18]:
output = chain.invoke({'query': 'Tell me joke abou the cat'})

In [19]:
print(output.content)

```json
{
  "setup": "Why was the cat sitting on the computer?",
  "punchline": "Because it wanted to keep an eye on the mouse!",
  "rating": 5
}
```


# Method2 - Using Parser directly

In [20]:
chain = prompt | llm | parser
output = chain.invoke({'query': 'Tell me a joke about the dogs'})
print(output)

setup='Why did the dog sit in the shade?' punchline="Because he didn't want to become a hot dog!" rating=5


# Method 3: Passing Pydantic Object directly to LLM

### Parsing with `.with_structured_output()` method
- This method takes a schema as input which specifies the names, types, and descriptions of the desired output attributes.
-  The schema can be specified as a TypedDict class, JSON Schema or a Pydantic class.


In [21]:
output = llm.invoke('Tell me a joke about the cat')
print(output.content)

Why was the cat sitting on the computer?

Because it wanted to keep an eye on the mouse!


In [22]:
structured_llm = llm.with_structured_output(Joke)

In [23]:
output= structured_llm.invoke('Tell me a joke about the cat')
print(output)

setup='Why was the cat sitting on the computer?' punchline='Because it wanted to keep an eye on the mouse!' rating=5


In [24]:
from langchain_openai import ChatOpenAI
llm2 = ChatOpenAI(temperature =0, model_name="gpt-4o")
structured_llm = llm2.with_structured_output(Joke)
output = structured_llm.invoke('Tell me a joke about the car')
print(output)

setup='Why did the car break up with the gas station?' punchline='Because it found someone more electrifying!' rating=4


### `JSON` Output Parser

- Output parsers accept a string or BaseMessage as input and can return an arbitrary type.


In [26]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=Joke)

In [27]:
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [29]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a joke. here is your formatting instruction.
    {format_instruction}
    
    Query: {query}
    Answer: ''',
    input_variables =['query'],
    partial_variables ={'format_instruction': parser.get_format_instructions()}
)


chain = prompt | llm
output= chain.invoke({'query': 'Tell me a joke abou the dog'})
print(output.content)

{"setup":"Why did the dog sit in the shade?","punchline":"Because he didn't want to become a hot dog!","rating":5}


In [30]:
chain = prompt |llm|parser
output= chain.invoke({'query': 'Tell me a joke abou the cat'})
print(output)

{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted to keep an eye on the mouse!', 'rating': 5}


### CSV Output Parser

- This output parser can be used when you want to return a list of comma-separated items.



In [31]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser =  CommaSeparatedListOutputParser()

print(parser.get_format_instructions())

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [35]:
format_instruction = parser.get_format_instructions()

prompt = PromptTemplate(
    template='''
    Answer the user query with a list of values. Here is your formatting instructions.
    {format_instruction}

    Query: {query}
    Answer: ''',
    input_variable=['query'],
    partial_variables ={'format_instruction': format_instruction}
)


In [36]:
chain = prompt | llm  | parser

output = chain.invoke({'query': 'generate my website seo keywords. I have content about the NLp, DL and LLM'})
print(output)

['NLP', 'DL', 'LLM', 'natural language processing', 'deep learning', 'large language models', 'machine learning', 'AI', 'artificial intelligence', 'text analysis', 'language understanding', 'neural networks', 'data science', 'predictive analytics', 'conversational AI', 'text generation']


### Datetime Output Parser

- Gives output in datetime format. Sometimes throws error if the LLM output is not in datetime format.

In [37]:
from langchain.output_parsers import DatetimeOutputParser

parser = DatetimeOutputParser()
#parser = DatetimeOutputParser(format="%d-%m-%y")

format_instruction = parser.get_format_instructions()

print(format_instruction)

Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 1993-05-12T04:38:51.649139Z, 1420-10-17T10:15:08.396461Z, 1144-06-25T15:31:17.033562Z

Return ONLY this string, no other words!


In [38]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a datetime. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': format_instruction}
)

In [40]:
chain = prompt | llm | parser

In [41]:
output = chain.invoke({'query': 'When the america got discovered'})
print(output)

1492-10-12 00:00:00
